# Webull HK ETF Downloader

Downloads Webull HK ETF historical bars into `data/raw/` using the normalized project schema: `symbol,timestamp,open,high,low,close,volume`. Webull uses leading-zero HK symbols (`02800`, `02828`, `07299`, `07568`, `07226`). The project configs use canonical symbols (`2800.HK`, etc.), so this notebook writes the canonical symbol into the CSV.

In [ ]:
# Run once if needed.
%pip install --upgrade webull-openapi-python-sdk pandas

In [ ]:
import logging
import os
import time
from datetime import date, datetime
from datetime import time as dt_time
from pathlib import Path
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import pandas as pd

logging.disable(logging.CRITICAL)

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def load_env(path: Path = ROOT / ".env") -> None:
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key.strip(), value)


def normalize_endpoint(endpoint: str) -> str:
    endpoint = endpoint.strip().rstrip("/")
    parsed = urlparse(endpoint)
    return parsed.netloc or endpoint


load_env()
required = ["WEBULL_APP_KEY", "WEBULL_APP_SECRET"]
missing = [key for key in required if not os.environ.get(key)]
if missing:
    raise RuntimeError(f"Missing required .env keys: {missing}")

print("Repo root:", ROOT)
print("Credentials loaded without displaying secrets.")


In [ ]:
from webull.core.client import ApiClient
from webull.data.common.category import Category
from webull.data.common.timespan import Timespan
from webull.data.data_client import DataClient

endpoint = normalize_endpoint(
    os.environ.get("WEBULL_API_ENDPOINT", "api.sandbox.webull.hk")
)
api_client = ApiClient(os.environ["WEBULL_APP_KEY"], os.environ["WEBULL_APP_SECRET"], "hk")
api_client.add_endpoint("hk", endpoint)
data_client = DataClient(api_client)

print("Initialized Webull data client.")
print("Endpoint:", endpoint)

In [ ]:
# Universe from the Quantphemes plan and lab work.
ETF_UNIVERSE = {
    "2800.HK": {"webull": "02800", "category": Category.HK_ETF.name},
    "2828.HK": {"webull": "02828", "category": Category.HK_ETF.name},
    "7299.HK": {"webull": "07299", "category": Category.HK_ETF.name},
    "7568.HK": {"webull": "07568", "category": Category.HK_ETF.name},
    "7226.HK": {"webull": "07226", "category": Category.HK_ETF.name},
}

# Choose one of: Timespan.M1.name, M5, M30, D.
# M30 is best for fast long-history Tier 2 training. M1 is precise but slow for years.
SYMBOLS_TO_DOWNLOAD = ["2800.HK", "2828.HK", "7299.HK", "7568.HK", "7226.HK"]
TIMESPAN = Timespan.M30.name
START_DATE = date(2018, 1, 1)
END_DATE = date.today()
COUNT_PER_REQUEST = "1200"  # SDK max.
REQUEST_SLEEP_SECONDS = 1.05  # Webull docs: 60 market-data requests/minute.

ETF_UNIVERSE

In [ ]:
HK = ZoneInfo("Asia/Hong_Kong")


def _millis(value: datetime) -> int:
    return int(value.timestamp() * 1000)


def normalize_bars(payload: list[dict], canonical_symbol: str) -> pd.DataFrame:
    if not payload:
        columns = ["symbol", "timestamp", "open", "high", "low", "close", "volume"]
        return pd.DataFrame(columns=columns)

    frame = pd.DataFrame(payload).rename(columns={"time": "timestamp"})
    frame["timestamp"] = pd.to_datetime(
        frame["timestamp"],
        utc=True,
        errors="coerce",
    ).dt.tz_convert(HK)
    frame["timestamp"] = frame["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
    frame["symbol"] = canonical_symbol

    for col in ["open", "high", "low", "close", "volume"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")

    columns = ["symbol", "timestamp", "open", "high", "low", "close", "volume"]
    return frame[columns].dropna().sort_values("timestamp")


def fetch_bars(canonical_symbol: str) -> pd.DataFrame:
    spec = ETF_UNIVERSE[canonical_symbol]
    start_dt = datetime.combine(START_DATE, dt_time.min, tzinfo=HK)
    cursor = datetime.combine(END_DATE, dt_time.max, tzinfo=HK)
    frames = []
    seen_oldest = set()

    while cursor >= start_dt:
        response = data_client.market_data.get_history_bar(
            spec["webull"],
            spec["category"],
            TIMESPAN,
            count=COUNT_PER_REQUEST,
            end_time=_millis(cursor),
        )
        if response.status_code != 200:
            raise RuntimeError(f"{canonical_symbol} returned HTTP {response.status_code}")

        payload = response.json()
        if not payload:
            break

        frame = normalize_bars(payload, canonical_symbol)
        frames.append(frame)
        oldest = pd.to_datetime(frame["timestamp"]).min().tz_localize(HK)
        newest = pd.to_datetime(frame["timestamp"]).max().tz_localize(HK)
        print(canonical_symbol, TIMESPAN, len(frame), "bars", oldest, "->", newest)

        if oldest in seen_oldest:
            break
        seen_oldest.add(oldest)
        if oldest <= start_dt:
            break

        cursor = oldest - pd.Timedelta(milliseconds=1)
        time.sleep(REQUEST_SLEEP_SECONDS)

    if not frames:
        return normalize_bars([], canonical_symbol)

    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(["symbol", "timestamp"]).sort_values("timestamp")
    out = out[pd.to_datetime(out["timestamp"]).dt.date >= START_DATE]
    out = out[pd.to_datetime(out["timestamp"]).dt.date <= END_DATE]
    return out.reset_index(drop=True)


In [ ]:
saved = []
for canonical_symbol in SYMBOLS_TO_DOWNLOAD:
    bars = fetch_bars(canonical_symbol)
    safe_symbol = canonical_symbol.replace(".", "_").lower()
    out = ROOT / "data" / "raw" / f"webull_{safe_symbol}_{TIMESPAN.lower()}.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    bars.to_csv(out, index=False)
    saved.append({"symbol": canonical_symbol, "rows": len(bars), "path": str(out)})
    print("Saved", len(bars), "rows to", out)

pd.DataFrame(saved)